In [8]:
import os
import pandas as pd

In [9]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
## PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTE

## svm
from sklearn.svm import SVC
import pandas as pd


In [10]:
ML_CLASSIFIER = ['MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", "XGBoost"] # "ELM"

def classify(X_train, y_train, X_test, y_test, classifier, search_type='grid'):
    """
    Classify the data using a Random Forest Classifier
    :param X_train: Training data
    :param y_train: Training labels
    :param X_test: Testing data
    :param y_test: Testing labels
    :param search_type: Type of search to perform
    :return: Accuracy of the classifier
    """
    # Create a Random Forest Classifier
    #clf = RandomForestClassifier()

    if classifier == "MLP":
        # clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000) ### uncomment if the performance is not good with below hyperparameter
        clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000)
    elif classifier == "GaussianNB": ### {'priors': None, 'var_smoothing': 1e-05}
        clf = GaussianNB(priors=None, var_smoothing= 1e-05) ### Often, the default parameters work well for many datasets.
    elif classifier == "Adaboost":
        dtc = DecisionTreeClassifier(ccp_alpha=0.0, class_weight='balanced', criterion='entropy', max_depth=10, max_features=None, max_leaf_nodes=None, min_impurity_decrease=0.0, min_samples_leaf=2, min_samples_split=2, splitter='best')
        clf = AdaBoostClassifier(estimator=dtc, n_estimators=200, random_state=0, learning_rate=1) ###{'learning_rate': 1, 'n_estimators': 200}
        
    elif classifier == "KNN":
        clf = KNeighborsClassifier(algorithm='auto', leaf_size=10, metric='euclidean', n_jobs=-1,  n_neighbors=1, p=1, weights='uniform')  
    elif classifier == "RFClassifier": ### 
        clf = RandomForestClassifier(bootstrap = False, criterion = 'entropy', max_depth = None, max_features = 'sqrt', min_samples_leaf = 1, min_samples_split = 2, n_estimators = 100, oob_score = False, random_state = 42)
    elif classifier == "SVM_linear": ##
        clf = SVC(kernel='linear')
    elif classifier == "SVM_sigmoid": ###
        clf = SVC(kernel='sigmoid')
    elif classifier == "SVM_RBF": 
        clf = SVC(C=10, cache_size = 200, class_weight = None, gamma = 'scale', kernel = 'rbf', max_iter = -1, probability = True, shrinking = True, tol = 0.001)
    elif classifier == "XGBoost": #'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.7
        clf = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,subsample=0.7)
    else:
        print("Invalid classifier")
        return None
        # Fit the model
    clf.fit(X_train, y_train)

    # Predict the test data
    y_pred = clf.predict(X_test)

    # Calculate the accuracy
   # accuracy = balanced_accuracy_score(y_test, y_pred)
    
    return y_pred


In [11]:
data = pd.read_csv('BT-small-2c-dataset_results_finetune_ALL_Models.csv')
data.head(26)

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,Average
0,resnet50,0.7371,0.7282,0.5976,0.6000,0.7387,0.6677,0.7282,0.4210,0.8177,0.6707
1,resnet101,0.6444,0.6782,0.7871,0.6250,0.7403,0.6516,0.6621,0.5137,0.6855,0.6653
2,densenet121,0.8089,0.8677,0.7726,0.7500,0.7387,0.8250,0.8677,0.8516,0.8427,0.8139
3,densenet169,0.8750,0.8677,0.8121,0.7750,0.8032,0.8500,0.8355,0.7944,0.8589,0.8302
4,vgg16,0.9589,0.9089,0.7355,0.8500,0.8048,0.8750,0.8516,0.9089,0.8927,0.8651
5,vgg19,0.8516,0.8605,0.7694,0.8589,0.8460,0.7766,0.8355,0.8516,0.8855,0.8373
6,alexnet,0.8677,0.9250,0.7460,0.8500,0.7726,0.8839,0.9339,0.7589,0.9500,0.8542
7,resnext50_32x4d,0.6371,0.7944,0.6315,0.6250,0.6065,0.6339,0.7532,0.5000,0.7516,0.6592
8,resnext101_32x8d,0.7839,0.8516,0.7226,0.6250,0.7298,0.7427,0.8266,0.6532,0.8339,0.7522
9,shufflenet_v2_x1_0,0.7855,0.9089,0.6589,0.7250,0.7637,0.7516,0.9089,0.9000,0.9000,0.8114


In [12]:
## sort the data by average
data = data.sort_values(by='Average', ascending=False)
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,Average
12,vit_base_patch16_224,1.0000,0.9750,0.8944,0.9250,0.9105,0.9250,0.9750,0.9339,0.9750,0.9460
19,vit_small_patch16_224,1.0000,0.9750,0.8282,0.9750,0.9016,0.9500,0.9339,0.9750,0.9750,0.9460
15,vit_small_patch32_224,0.9427,0.9500,0.9105,0.9589,0.9105,0.9339,0.9500,0.9589,0.9750,0.9434
20,vit_base_patch16_384,0.9339,0.9589,0.9032,0.9750,0.8855,0.9500,0.9750,0.9339,0.9589,0.9416
18,vit_tiny_patch16_224,0.9750,0.9750,0.8944,0.9500,0.9032,0.9250,0.8516,0.9427,0.9750,0.9324
22,vit_small_patch32_384,0.8677,0.9589,0.9089,0.9250,0.9427,0.9339,0.9355,0.9339,0.9266,0.9259
24,vit_base_patch32_384,0.9000,0.9750,0.8137,0.9250,0.9016,0.9250,0.9750,0.9339,0.9750,0.9249
13,vit_base_patch32_224,0.9339,0.9500,0.8460,0.9089,0.9355,0.9589,0.9016,0.9339,0.9339,0.9225
17,vit_base_patch8_224,0.9250,0.9500,0.8782,0.9250,0.8855,0.9000,0.9339,0.9339,0.9500,0.9202
23,vit_small_patch16_384,0.9500,0.9339,0.8766,0.9250,0.8121,0.9177,0.9177,0.9589,0.9750,0.9185


In [13]:
## change the indeces of the data to Model
data = data.set_index('Model')
data

,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,Average
Model,,,,,,,,,,
vit_base_patch16_224,1.0000,0.9750,0.8944,0.9250,0.9105,0.9250,0.9750,0.9339,0.9750,0.9460
vit_small_patch16_224,1.0000,0.9750,0.8282,0.9750,0.9016,0.9500,0.9339,0.9750,0.9750,0.9460
vit_small_patch32_224,0.9427,0.9500,0.9105,0.9589,0.9105,0.9339,0.9500,0.9589,0.9750,0.9434
vit_base_patch16_384,0.9339,0.9589,0.9032,0.9750,0.8855,0.9500,0.9750,0.9339,0.9589,0.9416
vit_tiny_patch16_224,0.9750,0.9750,0.8944,0.9500,0.9032,0.9250,0.8516,0.9427,0.9750,0.9324
vit_small_patch32_384,0.8677,0.9589,0.9089,0.9250,0.9427,0.9339,0.9355,0.9339,0.9266,0.9259
vit_base_patch32_384,0.9000,0.9750,0.8137,0.9250,0.9016,0.9250,0.9750,0.9339,0.9750,0.9249
vit_base_patch32_224,0.9339,0.9500,0.8460,0.9089,0.9355,0.9589,0.9016,0.9339,0.9339,0.9225
vit_base_patch8_224,0.9250,0.9500,0.8782,0.9250,0.8855,0.9000,0.9339,0.9339,0.9500,0.9202


In [14]:
data = data.sort_values(by = 'Average', axis = 1) 
data

,GaussianNB,KNN,Adaboost,SVM_sigmoid,RFClassifier,XGBoost,SVM_linear,SVM_RBF,MLP,Average
Model,,,,,,,,,,
vit_base_patch16_224,0.8944,0.9105,0.9250,0.9339,0.9250,1.0000,0.9750,0.9750,0.9750,0.9460
vit_small_patch16_224,0.8282,0.9016,0.9750,0.9750,0.9500,1.0000,0.9339,0.9750,0.9750,0.9460
vit_small_patch32_224,0.9105,0.9105,0.9589,0.9589,0.9339,0.9427,0.9500,0.9750,0.9500,0.9434
vit_base_patch16_384,0.9032,0.8855,0.9750,0.9339,0.9500,0.9339,0.9750,0.9589,0.9589,0.9416
vit_tiny_patch16_224,0.8944,0.9032,0.9500,0.9427,0.9250,0.9750,0.8516,0.9750,0.9750,0.9324
vit_small_patch32_384,0.9089,0.9427,0.9250,0.9339,0.9339,0.8677,0.9355,0.9266,0.9589,0.9259
vit_base_patch32_384,0.8137,0.9016,0.9250,0.9339,0.9250,0.9000,0.9750,0.9750,0.9750,0.9249
vit_base_patch32_224,0.8460,0.9355,0.9089,0.9339,0.9589,0.9339,0.9016,0.9339,0.9500,0.9225
vit_base_patch8_224,0.8782,0.8855,0.9250,0.9339,0.9000,0.9250,0.9339,0.9500,0.9500,0.9202


In [15]:
list(data.head(5).index)

['vit_base_patch16_224',
 'vit_small_patch16_224',
 'vit_small_patch32_224',
 'vit_base_patch16_384',
 'vit_tiny_patch16_224']

In [16]:
list(data.columns[-4:-1])

['SVM_linear', 'SVM_RBF', 'MLP']

### Top 3 performance model

In [17]:
top_ml_model_combinations = [
                          ['SVM_linear', 'SVM_RBF'],
                          ['SVM_linear', 'MLP'],
                          ['SVM_RBF', 'MLP'],
                          ['SVM_linear', "SVM_RBF", "MLP"]
                          ]

Ensemble vgg16 with Top 2 networks ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']

In [33]:



# top_model_combinations = [
#                           ['vit_base_patch32_384', 'vit_small_patch32_224'],
#                           ['vit_base_patch32_384', 'vgg16'],
#                           ['vit_small_patch32_224', 'vgg16'],
#                           ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']
#                           ]

In [19]:
columns = ['Model', 'vit_base_patch16_224',
                    'vit_small_patch16_224',
                    'vit_small_patch32_224',
                    'vit_base_patch16_384',
                    'vit_tiny_patch16_224']

dataframe = pd.DataFrame(columns=columns) ### new empty pandas DataFrame with specified column names
# add 12 rows to the dataframe with zero values 
for model_list in top_ml_model_combinations:
    model_name = ' + '.join(model_list)
    new_row = {}
    new_row['Model'] = model_name
    for col in columns[1:]:
        new_row[col] = 0
    dataframe.loc[len(dataframe)] = new_row

model_versions = ['simple_version', "with_normalization_&_PCA", "with_SMOTE_only", "with_normalization_&_PCA_SMOTE"]

model_version_index = 3
main_path = 'extracted_features_BT-small-2c'
for feature_extractor in columns[1:]:
    
    print('Deep Feature Extractor List:', feature_extractor)
    
    sub_dir = os.path.join(main_path, feature_extractor)
    X_train = np.load(os.path.join(sub_dir, 'train_data_array_features.npy'))
    y_train = np.load(os.path.join(sub_dir, 'train_data_array_labels.npy'))
    X_test = np.load(os.path.join(sub_dir, 'test_data_array_features.npy'))
    y_test = np.load(os.path.join(sub_dir, 'test_data_array_labels.npy'))

    X_train = np.squeeze(X_train, axis=1)
    X_test = np.squeeze(X_test, axis=1)

    # # apply the standard scaler and PCA
    scaler = StandardScaler()
    X_train_normalized = scaler.fit_transform(X_train)
    X_test_normalized = scaler.transform(X_test)

    # Step 2 & 3: Apply PCA
    n_components = int(0.45 * X_train.shape[1])
    pca = PCA(n_components=n_components)
    X_train = pca.fit_transform(X_train_normalized)
    X_test = pca.transform(X_test_normalized)

    ## no of example in each class
    print("no of example in each class before SMOTE:", np.bincount(y_train))


    # # ## over sample the data (data augmentation)
    smote = SMOTE(sampling_strategy='minority')
    X_train, y_train = smote.fit_resample(X_train, y_train)

    # ## no of example in each class
    print("no of example in each class after SMOTE:", np.bincount(y_train))
    
    for model_list in top_ml_model_combinations:
        all_predicted_probs = []
        for ml_classifier in model_list:
            probs = classify(X_train, y_train, X_test, y_test, ml_classifier)
            all_predicted_probs.append(probs)
        
        ## take the average of the probabilities
        all_predicted_probs = np.array(all_predicted_probs)
        all_predicted_probs = np.mean(all_predicted_probs, axis=0)
        ## if prob is greater than 0.5 then 1 else 0
        y_pred = np.where(all_predicted_probs >= 0.5, 1, 0)
        accuracy = balanced_accuracy_score(y_test, y_pred)
        accuracy = float(round(accuracy, 4))
        print('Accuracy:', accuracy)
    
        model_name = ' + '.join(model_list)

        dataframe.loc[dataframe['Model'] == model_name, feature_extractor] = accuracy

    print(dataframe)
    dataframe.to_csv(f'BT-small-2c-dataset_results_top_three_{model_versions[model_version_index]}_ml_ensemble.csv', index=False)


Deep Feature Extractor List: vit_base_patch16_224
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.975
Accuracy: 0.975


C:\Users\user\AppData\Local\Temp\ipykernel_5868\301564585.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.975' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, feature_extractor] = accuracy


Accuracy: 0.95
Accuracy: 0.975
                        Model  vit_base_patch16_224  vit_small_patch16_224  \
0        SVM_linear + SVM_RBF                 0.975                      0   
1            SVM_linear + MLP                 0.975                      0   
2               SVM_RBF + MLP                 0.950                      0   
3  SVM_linear + SVM_RBF + MLP                 0.975                      0   

   vit_small_patch32_224  vit_base_patch16_384  vit_tiny_patch16_224  
0                      0                     0                     0  
1                      0                     0                     0  
2                      0                     0                     0  
3                      0                     0                     0  
Deep Feature Extractor List: vit_small_patch16_224
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.95


C:\Users\user\AppData\Local\Temp\ipykernel_5868\301564585.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.95' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, feature_extractor] = accuracy


Accuracy: 0.95
Accuracy: 0.975
Accuracy: 0.975
                        Model  vit_base_patch16_224  vit_small_patch16_224  \
0        SVM_linear + SVM_RBF                 0.975                  0.950   
1            SVM_linear + MLP                 0.975                  0.950   
2               SVM_RBF + MLP                 0.950                  0.975   
3  SVM_linear + SVM_RBF + MLP                 0.975                  0.975   

   vit_small_patch32_224  vit_base_patch16_384  vit_tiny_patch16_224  
0                      0                     0                     0  
1                      0                     0                     0  
2                      0                     0                     0  
3                      0                     0                     0  
Deep Feature Extractor List: vit_small_patch32_224
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.975


...

Accuracy: 0.975
Accuracy: 0.975
Accuracy: 0.975
                        Model  vit_base_patch16_224  vit_small_patch16_224  \
0        SVM_linear + SVM_RBF                 0.975                  0.950   
1            SVM_linear + MLP                 0.975                  0.950   
2               SVM_RBF + MLP                 0.950                  0.975   
3  SVM_linear + SVM_RBF + MLP                 0.975                  0.975   

   vit_small_patch32_224  vit_base_patch16_384  vit_tiny_patch16_224  
0                  0.975                     0                     0  
1                  0.975                     0                     0  
2                  0.975                     0                     0  
3                  0.975                     0                     0  
Deep Feature Extractor List: vit_base_patch16_384
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.975
Accuracy: 0.975


C:\Users\user\AppData\Local\Temp\ipykernel_5868\301564585.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.975' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, feature_extractor] = accuracy


Accuracy: 0.975
Accuracy: 0.9589
                        Model  vit_base_patch16_224  vit_small_patch16_224  \
0        SVM_linear + SVM_RBF                 0.975                  0.950   
1            SVM_linear + MLP                 0.975                  0.950   
2               SVM_RBF + MLP                 0.950                  0.975   
3  SVM_linear + SVM_RBF + MLP                 0.975                  0.975   

   vit_small_patch32_224  vit_base_patch16_384  vit_tiny_patch16_224  
0                  0.975                0.9750                     0  
1                  0.975                0.9750                     0  
2                  0.975                0.9750                     0  
3                  0.975                0.9589                     0  
Deep Feature Extractor List: vit_tiny_patch16_224
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.925


C:\Users\user\AppData\Local\Temp\ipykernel_5868\301564585.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.925' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, feature_extractor] = accuracy


Accuracy: 0.925
Accuracy: 0.975
Accuracy: 0.95
                        Model  vit_base_patch16_224  vit_small_patch16_224  \
0        SVM_linear + SVM_RBF                 0.975                  0.950   
1            SVM_linear + MLP                 0.975                  0.950   
2               SVM_RBF + MLP                 0.950                  0.975   
3  SVM_linear + SVM_RBF + MLP                 0.975                  0.975   

   vit_small_patch32_224  vit_base_patch16_384  vit_tiny_patch16_224  
0                  0.975                0.9750                 0.925  
1                  0.975                0.9750                 0.925  
2                  0.975                0.9750                 0.975  
3                  0.975                0.9589                 0.950  


In [43]:
X_train.shape

(8536, 460)

In [42]:
all_predicted_probs.shape

(600,)

In [19]:
n_components

768

In [14]:
dataframe

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF
0,vit_base_patch32_384 + vit_small_patch32_224,0.736086,0.762759,0.334131,0.731610,0.771137,0.738784,0.799189,0.683773,0.754797
1,vit_base_patch32_384 + vit_small_patch16_224,0.733586,0.758041,0.333893,0.754932,0.796542,0.748919,0.779797,0.658766,0.754797
2,vit_small_patch32_224 + vit_small_patch16_224,0.740533,0.756554,0.410204,0.764571,0.812421,0.777488,0.719953,0.619690,0.765811
3,vit_base_patch32_384 + vit_small_patch32_224 +...,0.721848,0.767432,0.325231,0.747432,0.802421,0.782297,0.771554,0.665940,0.762297
